# 2026-06-19 Day 7: Benchmark, Report, Portfolio Gate

Goal: understand what to measure, how to profile, and how to turn the week into a report-grade artifact.


## How To Use This Reading Notebook

Use this before touching implementation for the day. The goal is not to memorize the paper. The goal is to extract the model of the system you need for today's code and notes.

Workflow:

1. Read only the assigned sections.
2. Fill the mini-lecture answer in your own words.
3. Open the repo files listed in the roadmap.
4. Run the listed checks or record why they skip.
5. Write a short exit ticket before moving on.


## Assigned Reading

| Source | Exact Target | Why It Matters Today |
| --- | --- | --- |
| [PyTorch profiler docs](https://docs.pytorch.org/docs/2.12/profiler.html) | `torch.profiler.profile`, `record_shapes`, `profile_memory` | Shows how to collect timing, shape, and memory evidence. |
| [FlashAttention](https://arxiv.org/abs/2205.14135) | Abstract and introduction | Explains why attention can be limited by memory movement, not only FLOPs. |


## Reading Summary

### PyTorch Profiler Docs

PyTorch profiler can collect CPU/GPU timing, shapes, stack traces, and memory information depending on configuration. For this project, the important options are shape recording and memory profiling because attention behavior changes with prompt length and cache mode.

Key takeaway for implementation: benchmark rows should include workload metadata, not just a speed number.

### FlashAttention Abstract And Introduction

FlashAttention argues that exact attention can be made faster and more memory-efficient by being IO-aware: reducing reads and writes between high-bandwidth memory and on-chip memory. This matters because attention often bottlenecks on memory movement, especially for long sequences.

Key takeaway for implementation: a tiny benchmark will not reproduce production kernels, but it should teach you which tensors and memory movements matter.


## Diagram Anchor

![Benchmark metrics dashboard](../../assets/figures/benchmark_metrics_dashboard.svg)

What to notice: tokens/sec without prompt length, cache mode, dtype, and hardware is not interpretable.

![Attention memory scaling](../../assets/figures/attention_memory_scaling.svg)

What to notice: the benchmark should separate full-prompt prefill from one-token decode.


## Repo Connection

Open these files after reading:

| File | What To Find |
| --- | --- |
| `reports/transformer_inference_report.md` | Report sections for research question, setup, results, limitations. |
| `configs/tiny_inference.json` | Default benchmark-style fields. |
| `results/` | Destination for raw JSON/CSV benchmark outputs. |
| `docs/project_01_combined_assignment_roadmap.md` | Final gate and expected benchmark command shape. |


## Benchmark Fields To Record Later

| Field | Why It Matters |
| --- | --- |
| timestamp | distinguishes runs |
| device and GPU | speed and memory depend on hardware |
| dtype | cache and activation memory depend on bytes per element |
| model config | parameters and shapes determine work |
| prompt length | controls prefill cost and cache size |
| new tokens | controls decode-loop duration |
| cache mode | cached and uncached runs answer different questions |
| prefill latency | time to first-token setup |
| decode latency or tokens/sec | streaming generation rate |
| peak memory | feasibility and capacity signal |


## Mini-Lecture Answer Seed

Attention can be memory-bound because forming and using attention matrices requires moving large intermediate tensors. FlashAttention improves exact attention by changing how the computation is tiled and how often data moves through memory hierarchy. KV cache attacks a different inference problem: it avoids recomputing old K/V tensors at every decode step. Benchmarking should separate these mechanisms rather than report a single aggregate tokens/sec number.


## Active Recall

1. What does prefill latency measure?
2. Why is prompt length required in every result row?
3. What does peak memory tell you that tokens/sec does not?
4. Why is cached-vs-uncached speedup not the same as production serving throughput?

## Exit Ticket

Fill a placeholder result table in `reports/transformer_inference_report.md` and list the next implementation gaps.
